# What Prompt Compression Removes: reproducibility companion
Kaggle API snapshot (2026-11-12): v2 submission 58201114 COMPLETE, 0.18; v3 submission 58250000 COMPLETE, 0.15.
This notebook (1) audits the official development tasks, (2) measures prompt bytes, (3) re-runs the instruction-unit audit of the v2 -> v3 compression,
(4) computes which evaluation sizes are consistent with the two displayed scores, and (5) validates and compiles the v3 bundle with the official wheelhouse.
It runs no model inference and makes no token, latency or significance claims. The API scores are historical evidence, not a fresh query.


In [ ]:
import json, hashlib, zipfile
from pathlib import Path
import numpy as np
V2_FILES = {'agent.yaml': 'name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Resolves a repository issue with a minimal, verified source patch.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - get_code_neighbors\n  - search_similar_code\n  - get_code_subgraph\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.2\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 7\n  max_turns: 80\n', 'prompts/system.md': "You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. Hidden tests were written from the issue specification, so exact symbol names, error messages, types, and edge cases in the issue are the contract.\n\n# HARD LIMITS\n- 28 tool calls and 7 minutes total (`submit_patch` and `get_status` do not consume tool calls). When either runs out, the working tree is graded as-is (`git add -N . && git diff HEAD`). An edit on disk counts; an empty tree always scores zero.\n- Never create, modify, or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`, `test/`, `testing/`) or runner configs (`pytest.ini`, `conftest.py`, `pyproject.toml`, `setup.cfg`, `tox.ini`, `sitecustomize.py`, `usercustomize.py`). Phase 2 resets them before grading.\n- Scratch scripts, test logs, and backups go strictly under `/tmp` via `run_command` (for example `cat > /tmp/repro.py << 'EOF'`). `read_file`/`write_file`/`edit_file` only accept paths inside `/workspace`, and any untracked file left under `/workspace` becomes part of your graded patch.\n- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always target one test file with `-x -q` and `-k` when possible, redirecting output to `/tmp` and inspecting the tail: `python3 -m pytest tests/path/test_x.py -x -q -k keyword > /tmp/test_1.log 2>&1; tail -n 35 /tmp/test_1.log`. Output of `run_command` is truncated at its FIRST 5,000 characters while pytest prints its verdict at the end.\n- Never call `search_similar_code`, `get_code_neighbors`, or `get_code_subgraph`. Code graphs omit async functions, and `search_similar_code` can exceed the 32,768-token context window and lose your patch.\n- Never look outside `/workspace` or install packages; the container is offline with all dependencies pre-installed.\n\n# CALL PLAN\n- Calls 1-8: locate and read focused line ranges.\n- Calls 9-12: edit SOURCE code. You MUST edit a source file by call 12. If uncertain, apply your best-reasoned edit in the most likely location rather than spending all calls reading. A wrong edit can be revised; pure reading scores zero.\n- Calls 13-20: verify with `/tmp` reproduction and targeted tests; refine across all parallel implementations.\n- Call 21 onward: no new exploration. Verify diff, check `git status --short && git diff`, and call `submit_patch`.\n\n# PROCEDURE\n\n## 1. Extract the Specification (no tool call)\nList from the issue: exact module, class, function, or CLI names; exact exception classes and message text; expected return types and behaviour; edge cases (None, empty inputs, unicode, negatives, defaults, concurrency/async). Treat literal strings and names in the issue as exact contracts.\n\n## 2. Locate Efficiently\n- Jump straight to paths named in the issue. When searching, batch known fixed-string searches in one `run_command` using `git grep -n -F` and `git ls-files` (`rg` and `tree` are not installed). Cap output short (`head -n 30`).\n- Read focused slices (40-120 lines) via `read_file` or `nl -ba FILE | sed -n 'START,ENDp'`. Inspect both the implementation and one existing caller or sibling test before editing.\n- Keep notes in brief plain text between tool calls: older tool outputs may be compacted on long trajectories, while your plain-text notes stay in context.\n\n## 3. Apply Surgical Source Edits\n- Prefer `edit_file` with a short, unique `old_string` verbatim from the file. Use `write_file` only if the issue genuinely requires a new source module.\n- Never repeat an identical failing edit. After two match failures, reread exact line numbers and apply one atomic Python replacement via `run_command` quoted heredoc asserting `content.count(old) == 1`.\n- Fix the root cause cleanly without breaking existing signatures or backward compatibility unless specified.\n- Propagate the fix to ALL parallel implementations where applicable: sync and async functions, `Client` and `AsyncClient`, alternate backends/transports, typing overloads, re-exports in `__init__.py` or `__all__`, and executable examples under `docs_src/` for documentation tasks.\n\n## 4. Verify Smartly (1-2 calls)\n- Run `python3 -m py_compile <edited_file>` and your `/tmp/repro.py` plus the single most relevant test file: `python3 -m pytest tests/test_x.py -x -q -k <kw> > /tmp/t.log 2>&1; tail -n 35 /tmp/t.log`.\n- Pre-existing failures from missing optional fixtures, missing network, or unrelated modules should be ignored; judge by the tests covering the modified code path.\n\n## 5. Clean Check and Submit\n- Run `git status --short && git diff` (`git diff` alone hides untracked files that `submit_patch` captures). Remove any stray `/workspace` scratch files and `git checkout --` any test/config edits before submitting.\n- Call `submit_patch` and confirm `patch_size > 0` and `files_changed > 0`, then conclude with a one-sentence summary of the fix.\n\n# DECISION RULES\n- Ambiguous specification: pick the most literal reading and edit; never stall or ask questions.\n- Multiple candidate sites: edit where wrong behaviour is produced rather than where it is merely observed.\n- Stuck after failed edits or empty searches: try one alternate symbol, then apply your best-reasoned small edit; never conclude no change is needed.\n\n# The issue, kept in view\n{problem_description}\n"}
V3_FILES = {'agent.yaml': 'name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Resolves a repository issue with a minimal, verified source patch.\ninstruction: !include prompts/system.md\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include configs/sampling.yaml\n', 'configs/sampling.yaml': 'temperature: 0.4\ntop_p: 0.95\ntop_k: 64\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n', 'eval_config.yaml': 'evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 28\n  max_time_minutes: 8\n  max_turns: 80\n', 'prompts/system.md': "You are an autonomous software engineer. Make the smallest correct fix to SOURCE code in /workspace so hidden tests pass. Treat exact symbol names, types, exception classes, and messages in the issue as literal contracts.\n\n# HARD LIMITS\n- Budget: 28 tool calls and 8 minutes (`submit_patch` and `get_status` are free). When either expires, `git add -N . && git diff HEAD` is graded; an unedited tree always scores 0.\n- Never create, edit, or delete test files (`test_*.py`, `*_test.py`, `tests/`, `test/`, `testing/`) or runner configs (`pytest.ini`, `conftest.py`, `pyproject.toml`, `setup.cfg`, `tox.ini`, `sitecustomize.py`, `usercustomize.py`).\n- Put scratch scripts and logs strictly under `/tmp` via `run_command`; any untracked file inside `/workspace` enters your graded patch.\n- Never run a full test suite (`pytest`, `pytest .`, `unittest discover`). Target one test file with `-x -q -k`, saving output to `/tmp`: `python3 -m pytest tests/path/test_x.py -x -q -k kw > /tmp/t.log 2>&1; tail -n 35 /tmp/t.log` (`run_command` keeps only the first 5,000 chars).\n- Never inspect outside `/workspace` or install packages (offline container).\n\n# CALL PLAN\n- Calls 1-8: locate and inspect exact target source + caller/test lines.\n- Calls 9-12: edit SOURCE code. You MUST make at least one source edit by call 12 even if uncertain; an imperfect patch can be refined, pure reading scores 0.\n- Calls 13-20: run `/tmp` repro + one targeted test file; fix edge cases across all parallel code paths.\n- Call 21+: stop exploring, verify `git status --short && git diff`, and call `submit_patch`.\n\n# WORKFLOW\n1. **Spec (no tool call)**: List exact files/symbols, error types/messages, return values, and edge cases (`None`, empty, unicode, bounds, defaults, async).\n2. **Locate**: Start with paths mentioned in the issue. Batch known fixed-string searches in one `run_command` using `git grep -n -F` and `git ls-files` (`head -n 30`). Read 40-120 line slices via `read_file` or `sed -n 'S,Ep'`, including one caller or sibling test. Keep brief plain-text notes between turns so key facts survive context compaction.\n3. **Edit**: Use `edit_file` with a short unique `old_string` verbatim from the file. Never repeat an identical failed edit; after 2 match failures, reread exact line numbers and apply one atomic Python replacement via `run_command` quoted heredoc asserting `content.count(old) == 1`. Propagate fixes across ALL parallel implementations (sync/async, `Client`/`AsyncClient`, alternate backends, typing overloads, `__init__.py`/`__all__`, `docs_src/`). Keep unrelated code and signatures intact.\n4. **Verify**: Run `python3 -m py_compile <edited_file>`, a `/tmp/repro.py` script, and the single closest test file via `/tmp/t.log`. Ignore pre-existing failures from missing optional fixtures or unrelated modules.\n5. **Submit**: Run `git status --short && git diff`, delete any stray `/workspace` files, revert accidental non-source edits with `git checkout --`, then call `submit_patch` and confirm `patch_size > 0` and `files_changed > 0`. Finish with a one-sentence summary.\n\n# DECISION RULES\n- Ambiguous spec: follow the most literal interpretation and act; never stall.\n- Stuck on empty searches: try one alternate symbol, then apply your best-reasoned small fix. Never conclude no edit is needed.\n\n# Issue\n{problem_description}\n"}

In [ ]:
candidates=sorted(Path('/kaggle/input/competitions/gemma-4-developer-agent').glob('**/tasks.jsonl'))
if not candidates:
    candidates=sorted(Path('/kaggle/input').glob('**/tasks.jsonl'))
assert len(candidates)==1, f'Need exactly one official tasks.jsonl; got {candidates}'
p=candidates[0]
raw=p.read_bytes()
tasks=[json.loads(line) for line in raw.decode().splitlines() if line.strip()]
problems=[str(t.get('problem_statement') or t.get('problem_description') or t.get('description') or '') for t in tasks]
assert tasks and all(problems), 'Missing problem statements'
audit={'tasks_path':str(p),'tasks_sha256':hashlib.sha256(raw).hexdigest(),'task_count':len(tasks),'problem_bytes_quantiles':np.percentile([len(x.encode()) for x in problems],[0,50,90,100]).tolist()}
print(json.dumps(audit,indent=2))


In [ ]:
rows=[]
for label,files in [('v2',V2_FILES),('v3',V3_FILES)]:
    prompt=files['prompts/system.md']
    sizes=[len(prompt.replace('{problem_description}',x).encode('utf-8')) for x in problems]
    rows.append({'variant':label,'prompt_template_bytes':len(prompt.encode('utf-8')),'expanded_prompt_bytes_p50':float(np.median(sizes)),'expanded_prompt_bytes_p90':float(np.percentile(sizes,90))})
reduction=rows[0]['prompt_template_bytes']-rows[1]['prompt_template_bytes']
print(json.dumps(rows,indent=2))
print('Measured template byte reduction:',reduction)
print('No tokenizer or serialized tool-schema measurements are claimed.')
report={'provenance':audit,'prompt_measurements':rows,'template_bytes_saved':reduction,'verified_historical_scores':{'58201114':0.18,'58250000':0.15},'score_delta':0.15-0.18,'limitations':['One evaluation per variant','Multiple configuration changes','No paired task outcomes','No causal solve-rate attribution','No inference executed by CPU companion']}
Path('/kaggle/working/evidence_report.json').write_text(json.dumps(report,indent=2))


## Instruction-unit audit
Each v2 guidance unit cites a verbatim v2 excerpt and, if it survives, a verbatim v3 excerpt. The audit fails if any excerpt is not found.


In [ ]:
"""Instruction-level audit of the v2 -> v3 system-prompt compression.

The v2 prompt is decomposed by hand into atomic guidance units. Each unit cites a
verbatim excerpt of v2 and, when the guidance survives, a verbatim excerpt of v3.
``audit_units`` refuses to run if any excerpt is not found in the corresponding
prompt, so the coding cannot silently drift from the submitted bytes.

Kinds:
  directive    - tells the model what to do or not do
  rationale    - explains why a directive exists
  environment  - states a fact about the sandbox or tools
  example      - a concrete illustration of a directive
"""


import re
from collections import Counter
from typing import Dict, List, NamedTuple, Optional


class Unit(NamedTuple):
    uid: str
    section: str
    kind: str
    summary: str
    v2_excerpt: str
    v3_excerpt: Optional[str]  # None means the guidance is absent from v3


UNITS: List[Unit] = [
    Unit("U01", "preamble", "directive", "Smallest correct source change",
         "smallest correct change to the SOURCE code", "smallest correct fix to SOURCE code"),
    Unit("U02", "preamble", "rationale", "Hidden tests derive from the issue text",
         "Hidden tests were written from the issue specification", None),
    Unit("U03", "preamble", "directive", "Treat issue names and messages as contracts",
         "edge cases in the issue are the contract", "as literal contracts"),
    Unit("U04", "hard limits", "environment", "Tool-call and time budget",
         "28 tool calls and 7 minutes total", "28 tool calls and 8 minutes"),
    Unit("U05", "hard limits", "environment", "submit_patch and get_status are free",
         "do not consume tool calls", "are free"),
    Unit("U06", "hard limits", "environment", "Working tree is graded when budget expires",
         "the working tree is graded as-is", "`git add -N . && git diff HEAD` is graded"),
    Unit("U07", "hard limits", "rationale", "An edit on disk counts even without submission",
         "An edit on disk counts", None),
    Unit("U08", "hard limits", "directive", "Never touch tests or runner configs",
         "Never create, modify, or delete test files", "Never create, edit, or delete test files"),
    Unit("U09", "hard limits", "rationale", "Test files are reset before grading",
         "Phase 2 resets them before grading", None),
    Unit("U10", "hard limits", "directive", "Scratch files only under /tmp",
         "Scratch scripts, test logs, and backups go strictly under `/tmp`",
         "Put scratch scripts and logs strictly under `/tmp`"),
    Unit("U11", "hard limits", "example", "Heredoc example for /tmp scripts",
         "for example `cat > /tmp/repro.py << 'EOF'`", None),
    Unit("U12", "hard limits", "environment", "File tools only accept /workspace paths",
         "only accept paths inside `/workspace`", None),
    Unit("U13", "hard limits", "environment", "Untracked workspace files enter the patch",
         "any untracked file left under `/workspace` becomes part of your graded patch",
         "any untracked file inside `/workspace` enters your graded patch"),
    Unit("U14", "hard limits", "directive", "Never run a full test suite",
         "Never run a full test suite", "Never run a full test suite"),
    Unit("U15", "hard limits", "directive", "Target one test file, log to /tmp, read the tail",
         "Always target one test file", "Target one test file"),
    Unit("U16", "hard limits", "environment", "run_command keeps only the first 5,000 characters",
         "truncated at its FIRST 5,000 characters", "keeps only the first 5,000 chars"),
    Unit("U17", "hard limits", "rationale", "pytest prints its verdict at the end",
         "pytest prints its verdict at the end", None),
    Unit("U18", "hard limits", "directive", "Never call the three graph tools",
         "Never call `search_similar_code`, `get_code_neighbors`, or `get_code_subgraph`", None),
    Unit("U19", "hard limits", "rationale", "Graph tools omit async code and can overflow context",
         "Code graphs omit async functions", None),
    Unit("U20", "hard limits", "directive", "Stay inside /workspace; install nothing",
         "Never look outside `/workspace` or install packages",
         "Never inspect outside `/workspace` or install packages"),
    Unit("U21", "hard limits", "environment", "Container is offline",
         "the container is offline", "offline container"),
    Unit("U22", "call plan", "directive", "Calls 1-8: locate and read",
         "Calls 1-8: locate and read focused line ranges", "Calls 1-8: locate and inspect"),
    Unit("U23", "call plan", "directive", "Edit source by call 12",
         "You MUST edit a source file by call 12", "You MUST make at least one source edit by call 12"),
    Unit("U24", "call plan", "rationale", "A wrong edit can be revised; reading alone scores zero",
         "pure reading scores zero", "pure reading scores 0"),
    Unit("U25", "call plan", "directive", "Calls 13-20: reproduce and test",
         "Calls 13-20: verify with `/tmp` reproduction and targeted tests",
         "Calls 13-20: run `/tmp` repro + one targeted test file"),
    Unit("U26", "call plan", "directive", "Call 21+: stop exploring and submit",
         "Call 21 onward: no new exploration", "Call 21+: stop exploring"),
    Unit("U27", "procedure", "directive", "Extract a specification before tool calls",
         "Extract the Specification (no tool call)", "**Spec (no tool call)**"),
    Unit("U28", "procedure", "directive", "Start from paths named in the issue",
         "Jump straight to paths named in the issue", "Start with paths mentioned in the issue"),
    Unit("U29", "procedure", "directive", "Batch fixed-string searches with git grep",
         "batch known fixed-string searches in one `run_command` using `git grep -n -F`",
         "Batch known fixed-string searches in one `run_command` using `git grep -n -F`"),
    Unit("U30", "procedure", "environment", "rg and tree are not installed",
         "`rg` and `tree` are not installed", None),
    Unit("U31", "procedure", "directive", "Read 40-120 line slices",
         "Read focused slices (40-120 lines)", "Read 40-120 line slices"),
    Unit("U32", "procedure", "directive", "Inspect one caller or sibling test before editing",
         "one existing caller or sibling test", "including one caller or sibling test"),
    Unit("U33", "procedure", "directive", "Keep brief plain-text notes",
         "Keep notes in brief plain text between tool calls", "Keep brief plain-text notes between turns"),
    Unit("U34", "procedure", "rationale", "Older tool outputs may be compacted",
         "older tool outputs may be compacted", "survive context compaction"),
    Unit("U35", "procedure", "directive", "Prefer edit_file with a short unique old_string",
         "Prefer `edit_file` with a short, unique `old_string`", "Use `edit_file` with a short unique `old_string`"),
    Unit("U36", "procedure", "directive", "Use write_file only for genuinely new modules",
         "Use `write_file` only if the issue genuinely requires a new source module", None),
    Unit("U37", "procedure", "directive", "Never repeat a failed edit; heredoc fallback",
         "Never repeat an identical failing edit", "Never repeat an identical failed edit"),
    Unit("U38", "procedure", "directive", "Fix the root cause",
         "Fix the root cause cleanly", None),
    Unit("U39", "procedure", "directive", "Preserve existing signatures",
         "without breaking existing signatures", "Keep unrelated code and signatures intact"),
    Unit("U40", "procedure", "directive", "Propagate fixes to all parallel implementations",
         "Propagate the fix to ALL parallel implementations", "Propagate fixes across ALL parallel implementations"),
    Unit("U41", "procedure", "directive", "py_compile, repro script and one test file",
         "Run `python3 -m py_compile <edited_file>`", "Run `python3 -m py_compile <edited_file>`"),
    Unit("U42", "procedure", "directive", "Ignore unrelated pre-existing failures",
         "Pre-existing failures from missing optional fixtures", "Ignore pre-existing failures from missing optional fixtures"),
    Unit("U43", "procedure", "directive", "Judge by tests covering the modified path",
         "judge by the tests covering the modified code path", None),
    Unit("U44", "procedure", "directive", "Check git status and diff before submitting",
         "Run `git status --short && git diff`", "Run `git status --short && git diff`"),
    Unit("U45", "procedure", "rationale", "git diff alone hides untracked files",
         "`git diff` alone hides untracked files", None),
    Unit("U46", "procedure", "directive", "Remove scratch files; revert test/config edits",
         "Remove any stray `/workspace` scratch files", "delete any stray `/workspace` files"),
    Unit("U47", "procedure", "directive", "Confirm a non-empty submitted patch",
         "confirm `patch_size > 0` and `files_changed > 0`", "confirm `patch_size > 0` and `files_changed > 0`"),
    Unit("U48", "decision rules", "directive", "Ambiguity: take the most literal reading",
         "pick the most literal reading and edit", "follow the most literal interpretation and act"),
    Unit("U49", "decision rules", "directive", "Edit where wrong behaviour is produced, not observed",
         "edit where wrong behaviour is produced rather than where it is merely observed", None),
    Unit("U50", "decision rules", "directive", "When stuck, try one alternate symbol then edit",
         "try one alternate symbol", "try one alternate symbol"),
    Unit("U51", "decision rules", "directive", "Never conclude no change is needed",
         "never conclude no change is needed", "Never conclude no edit is needed"),
]

# Labelling choices that a different rater could reasonably make differently.
# Environment facts that also explain a nearby rule (U13 explains why scratch files
# belong in /tmp, U16 why test output is read from its tail, U21 why nothing is installed).
BORDERLINE_RATIONALES = ("U13", "U16", "U21")
# Units that v3 keeps in part: U07 (v3 keeps "an unedited tree scores 0" but not
# "an edit on disk counts"), U43 (v3 keeps "ignore pre-existing failures" but not
# "judge by the tests covering the modified path").
PARTLY_RETAINED = ("U07", "U43")
# Guidance about tools that v3 removed, so it became moot rather than lost.
MOOT_AFTER_TOOL_REMOVAL = ("U18", "U19")

SECTION_ALIASES = {
    "hard limits": "hard limits",
    "call plan": "call plan",
    "procedure": "procedure",
    "workflow": "procedure",
    "decision rules": "decision rules",
    "the issue, kept in view": "issue",
    "issue": "issue",
}


def section_bytes(prompt: str) -> Dict[str, int]:
    """UTF-8 bytes per top-level ``# `` section; text before the first heading is 'preamble'."""
    sizes: Dict[str, int] = Counter()
    current = "preamble"
    for line in prompt.splitlines(keepends=True):
        match = re.match(r"# (?!#)(.+)", line)
        if match:
            current = SECTION_ALIASES[match.group(1).strip().lower()]
        sizes[current] += len(line.encode("utf-8"))
    return dict(sizes)


def _rates(units, kinds, dropped, excluded=()):
    out = {}
    for kind in ("rationale", "directive"):
        ids = [u.uid for u in units if kinds[u.uid] == kind and u.uid not in excluded]
        gone = [i for i in ids if i in dropped and i not in excluded]
        out[kind] = {"dropped": len(gone), "total": len(ids), "rate": round(len(gone) / len(ids), 3)}
    out["rate_ratio"] = round(out["rationale"]["rate"] / out["directive"]["rate"], 2)
    return out


def sensitivity(units: List[Unit] = UNITS) -> Dict[str, object]:
    """How the headline (rationales dropped far more often than directives) responds to
    defensible alternative labellings, and to leaving out any single unit."""
    kinds = {u.uid: u.kind for u in units}
    dropped = {u.uid for u in units if u.v3_excerpt is None}
    reclassified = {**kinds, **{i: "rationale" for i in BORDERLINE_RATIONALES}}
    still_dropped = dropped - set(PARTLY_RETAINED)
    scenarios = {
        "as_labelled": _rates(units, kinds, dropped),
        "borderline_environment_facts_as_rationales": _rates(units, reclassified, dropped),
        "partly_retained_units_counted_as_kept": _rates(units, kinds, still_dropped),
        "both_changes": _rates(units, reclassified, still_dropped),
        "graph_tool_units_excluded": _rates(units, kinds, dropped, MOOT_AFTER_TOOL_REMOVAL),
    }
    leave_one_out = [_rates([u for u in units if u.uid != x.uid], kinds, dropped) for x in units]
    ratios = [s["rate_ratio"] for s in scenarios.values()]
    return {
        "scenarios": scenarios,
        "rate_ratio_range_scenarios": [min(ratios), max(ratios)],
        "rate_ratio_range_leave_one_out": [min(r["rate_ratio"] for r in leave_one_out),
                                           max(r["rate_ratio"] for r in leave_one_out)],
    }


def audit_units(v2_prompt: str, v3_prompt: str, units: List[Unit] = UNITS) -> Dict[str, object]:
    """Validates every excerpt against the prompts and summarises what v3 dropped."""
    missing = [u.uid for u in units if u.v2_excerpt not in v2_prompt]
    missing += [u.uid for u in units if u.v3_excerpt is not None and u.v3_excerpt not in v3_prompt]
    if missing:
        raise ValueError(f"excerpts not found verbatim in the prompts: {missing}")
    ids = [u.uid for u in units]
    if len(ids) != len(set(ids)):
        raise ValueError("duplicate unit ids")

    dropped = [u for u in units if u.v3_excerpt is None]
    by_kind = {
        kind: {"v2": sum(u.kind == kind for u in units), "dropped": sum(u.kind == kind for u in dropped)}
        for kind in ("directive", "rationale", "environment", "example")
    }
    return {
        "units_total": len(units),
        "units_retained": len(units) - len(dropped),
        "units_dropped": len(dropped),
        "by_kind": by_kind,
        "dropped": [{"id": u.uid, "kind": u.kind, "section": u.section, "summary": u.summary} for u in dropped],
        "units": [{"id": u.uid, "section": u.section, "kind": u.kind, "dropped": u.v3_excerpt is None} for u in units],
        "sensitivity": sensitivity(units),
        "section_bytes": {"v2": section_bytes(v2_prompt), "v3": section_bytes(v3_prompt)},
    }


In [ ]:
audit=audit_units(V2_FILES['prompts/system.md'],V3_FILES['prompts/system.md'])
print(json.dumps({k:audit[k] for k in ('units_total','units_retained','units_dropped','by_kind','section_bytes')},indent=2))
for u in audit['dropped']: print(u['id'],u['kind'],'-',u['summary'])
print('Sensitivity of the rationale-vs-directive contrast to alternative labellings:')
for name,sc in audit['sensitivity']['scenarios'].items():
    print(' ',name,'| rationales dropped',sc['rationale']['dropped'],'/',sc['rationale']['total'],'| directives dropped',sc['directive']['dropped'],'/',sc['directive']['total'],'| rate ratio',sc['rate_ratio'])
print('Rate-ratio range across scenarios:',audit['sensitivity']['rate_ratio_range_scenarios'],'| leave-one-unit-out:',audit['sensitivity']['rate_ratio_range_leave_one_out'])


## Score resolution
Assuming binary per-task outcomes and two-decimal rounding, which evaluation sizes N reproduce both displayed scores, and how many tasks does the gap represent?


In [ ]:
"""What two rounded leaderboard scores can and cannot tell us.

Assumption (stated, not verified): each hidden task contributes a binary pass/fail
and the public score is the pass fraction rounded half-up to two decimals.
Under that assumption we enumerate every evaluation size N whose integer pass
counts reproduce both displayed scores, and the task-count gaps they imply.
"""


import math
from decimal import ROUND_HALF_UP, Decimal
from typing import Dict, List, Tuple


def displayed(passes: int, n: int, places: int = 2) -> float:
    quantum = Decimal(1).scaleb(-places)
    return float((Decimal(passes) / Decimal(n)).quantize(quantum, rounding=ROUND_HALF_UP))


def pass_counts(score: float, n: int) -> List[int]:
    """Integer pass counts k for which k/N displays as ``score``."""
    return [k for k in range(n + 1) if displayed(k, n) == score]


def consistent_sizes(a: float, b: float, n_min: int = 1, n_max: int = 300) -> Dict[int, Tuple[List[int], List[int]]]:
    """Evaluation sizes N at which both scores are attainable, with their pass counts."""
    out = {}
    for n in range(n_min, n_max + 1):
        ka, kb = pass_counts(a, n), pass_counts(b, n)
        if ka and kb:
            out[n] = (ka, kb)
    return out


def task_gap_range(ka: List[int], kb: List[int]) -> Tuple[int, int]:
    gaps = [x - y for x in ka for y in kb]
    return min(gaps), max(gaps)


def required_n_independent(p: float, delta: float, z: float = 1.96) -> int:
    """Smallest N at which a difference ``delta`` between two independent binomial
    proportions near ``p`` exceeds ``z`` standard errors: delta > z*sqrt(2p(1-p)/N)."""
    return math.ceil(2 * p * (1 - p) * (z / delta) ** 2)


def summarize(a: float = 0.18, b: float = 0.15, n_max: int = 300) -> Dict[str, object]:
    sizes = consistent_sizes(a, b, 1, n_max)
    inconsistent_small = [n for n in range(1, 130) if n not in sizes]
    gaps = {n: task_gap_range(*sizes[n]) for n in sizes}
    up_to_129 = [g for n, g in gaps.items() if n <= 129]
    return {
        "assumption": "binary per-task outcomes; score = pass fraction rounded half-up to 2 decimals",
        "scores": [a, b],
        "smallest_consistent_n": min(sizes),
        "consistent_n_count_up_to": {"n_max": n_max, "count": len(sizes)},
        "inconsistent_n_up_to_129": inconsistent_small,
        "task_gap_min_max_for_n_le_129": [min(g[0] for g in up_to_129), max(g[1] for g in up_to_129)],
        "gap_by_n": {str(n): list(g) for n, g in gaps.items()},
        "required_n_two_sigma_independent": required_n_independent((a + b) / 2, a - b),
    }


In [ ]:
res=summarize(0.18,0.15)
gap=res.pop('gap_by_n')
print(json.dumps(res,indent=2))
print('Gap (tasks) at selected N:',{n:gap[str(n)] for n in (33,62,100,120,129) if str(n) in gap})
report.update(instruction_audit=audit,score_resolution=res)
Path('/kaggle/working/evidence_report.json').write_text(json.dumps(report,indent=2))


## Figures and configuration checksums
Regenerates the paper's Figures 1 and 2 from the values computed above, and identifies each configuration by a SHA-256 over its file contents (ZIP bytes vary with the DEFLATE implementation, so they are not used as identity).


In [ ]:
from typing import Mapping

BLUE, AMBER, GREY = ('#2f6fdb', '#d08a00', '#6b7280')
SECTIONS = ['preamble', 'hard limits', 'call plan', 'procedure', 'decision rules', 'issue']
KIND_COLORS = {'directive': '#2f6fdb', 'rationale': '#d08a00', 'environment': '#1b9e77', 'example': '#7b5ea7'}
MAP_SECTIONS = ['preamble', 'hard limits', 'call plan', 'procedure', 'decision rules']

def _unit_map(ax, audit: dict) -> None:
    """One square per v2 guidance unit: colour = kind, solid = kept in v3, hatched outline = absent."""
    from matplotlib.patches import Patch, Rectangle

    for row, section in enumerate(MAP_SECTIONS):
        units = [u for u in audit["units"] if u["section"] == section]
        for col, u in enumerate(units):
            color = KIND_COLORS[u["kind"]]
            if u["dropped"]:
                ax.add_patch(Rectangle((col, row), 0.86, 0.8, facecolor="white", edgecolor=color,
                                       hatch="////", linewidth=1.4))
            else:
                ax.add_patch(Rectangle((col, row), 0.86, 0.8, facecolor=color, edgecolor=color))
    ax.set_xlim(0, 22)
    ax.set_ylim(len(MAP_SECTIONS), -0.1)
    ax.set_yticks([i + 0.4 for i in range(len(MAP_SECTIONS))], MAP_SECTIONS)
    ax.set_xticks([])
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.tick_params(left=False)
    ax.set_title("(b) The 51 v2 guidance units: kept or absent in v3", loc="left")
    handles = [Patch(facecolor=c, edgecolor=c, label=k) for k, c in KIND_COLORS.items()]
    handles.append(Patch(facecolor="white", edgecolor="#444", hatch="////", label="absent from v3"))
    ax.legend(handles=handles, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=5,
              fontsize=8, handlelength=1.2, columnspacing=1.0)


def make_figures(report: dict, out_dir: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
    out_dir.mkdir(parents=True, exist_ok=True)

    audit = report["instruction_audit"]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1, 1.25]})
    sb = audit["section_bytes"]
    y = range(len(SECTIONS))
    ax1.barh([i + 0.2 for i in y], [sb["v2"].get(s, 0) for s in SECTIONS], 0.4, color="#a7adba", label="v2 (5,351 B)")
    ax1.barh([i - 0.2 for i in y], [sb["v3"].get(s, 0) for s in SECTIONS], 0.4, color="#2b3140", label="v3 (3,340 B)")
    ax1.set_yticks(list(y), SECTIONS)
    ax1.invert_yaxis()
    ax1.set_xlabel("UTF-8 bytes")
    ax1.set_title("(a) Prompt bytes by section", loc="left")
    ax1.legend(frameon=False, loc="lower right")

    _unit_map(ax2, audit)
    fig.tight_layout()
    fig.savefig(out_dir / "fig1_prompt_audit.png", dpi=200)
    plt.close(fig)

    sr = report["score_resolution"]
    ns = sorted(int(n) for n in sr["gap_by_n"])
    lo = [sr["gap_by_n"][str(n)][0] for n in ns]
    hi = [sr["gap_by_n"][str(n)][1] for n in ns]
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.vlines(ns, lo, hi, color=BLUE, linewidth=1.2)
    ax.scatter(ns, lo, s=6, color=BLUE)
    ax.scatter(ns, hi, s=6, color=BLUE)
    ax.axvline(129, color=GREY, linestyle="--", linewidth=0.8)
    ax.text(131, max(hi) - 0.5, "129 public dev tasks", color=GREY, fontsize=8)
    ax.set_xlabel("Hypothetical number of evaluated tasks N (only N reproducing both 0.18 and 0.15 shown)")
    ax.set_ylabel("v2 - v3 gap (tasks)")
    ax.set_title("Task-count gap implied by displayed scores 0.18 and 0.15 (binary scoring assumed)", loc="left")
    ax.set_xlim(0, max(ns) + 5)
    fig.tight_layout()
    fig.savefig(out_dir / "fig2_score_resolution.png", dpi=200)
    plt.close(fig)


def content_digest(files: Mapping[str, str]) -> str:
    """SHA-256 over sorted (path, bytes) pairs.

    Unlike a ZIP hash, this does not depend on the DEFLATE implementation
    (e.g. zlib vs zlib-ng), so it identifies a configuration across Python builds.
    """
    import hashlib

    h = hashlib.sha256()
    for name in sorted(files):
        data = files[name].encode("utf-8")
        h.update(name.encode("utf-8") + b"\0" + len(data).to_bytes(8, "big") + data)
    return h.hexdigest()


In [ ]:
from IPython.display import Image, display
fig_dir=Path('/kaggle/working/figures')
make_figures({'instruction_audit':audit,'score_resolution':{**res,'gap_by_n':gap}},fig_dir)
for name in ('fig1_prompt_audit.png','fig2_score_resolution.png'):
    display(Image(filename=str(fig_dir/name)))
digests={'v2':content_digest(V2_FILES),'v3':content_digest(V3_FILES)}
print(json.dumps(digests,indent=2))
report.update(bundle_content_digest=digests)
Path('/kaggle/working/evidence_report.json').write_text(json.dumps(report,indent=2))


## Configuration validation
The following cells reproduce the v3 bundle and run the official compiler. The tool implementations used for compilation are signature stubs, not executed repair tools. Compilation success establishes configuration compatibility only.


In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
import yaml

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("working")
WORKING.mkdir(parents=True, exist_ok=True)
BUNDLE = WORKING / "submission_bundle"
shutil.rmtree(BUNDLE, ignore_errors=True)
BUNDLE.mkdir(parents=True, exist_ok=True)

SUBMISSION_FILES = json.loads('{\n  "agent.yaml": "name: swe_agent\\nmodel: gemma-4-31b-it-qat-w4a16-ct\\ndescription: Resolves a repository issue with a minimal, verified source patch.\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - read_file\\n  - edit_file\\n  - write_file\\n  - get_status\\n  - submit_patch\\ngenerate_content_config: !include configs/sampling.yaml\\n",\n  "configs/sampling.yaml": "temperature: 0.4\\ntop_p: 0.95\\ntop_k: 64\\nmax_output_tokens: 8192\\nthinking_config:\\n  thinking_budget: 4096\\n  include_thoughts: true\\n",\n  "eval_config.yaml": "evaluation:\\n  timeout_seconds: 240\\n  max_tool_calls: 28\\n  max_time_minutes: 8\\n  max_turns: 80\\n",\n  "prompts/system.md": "You are an autonomous software engineer. Make the smallest correct fix to SOURCE code in /workspace so hidden tests pass. Treat exact symbol names, types, exception classes, and messages in the issue as literal contracts.\\n\\n# HARD LIMITS\\n- Budget: 28 tool calls and 8 minutes (`submit_patch` and `get_status` are free). When either expires, `git add -N . && git diff HEAD` is graded; an unedited tree always scores 0.\\n- Never create, edit, or delete test files (`test_*.py`, `*_test.py`, `tests/`, `test/`, `testing/`) or runner configs (`pytest.ini`, `conftest.py`, `pyproject.toml`, `setup.cfg`, `tox.ini`, `sitecustomize.py`, `usercustomize.py`).\\n- Put scratch scripts and logs strictly under `/tmp` via `run_command`; any untracked file inside `/workspace` enters your graded patch.\\n- Never run a full test suite (`pytest`, `pytest .`, `unittest discover`). Target one test file with `-x -q -k`, saving output to `/tmp`: `python3 -m pytest tests/path/test_x.py -x -q -k kw > /tmp/t.log 2>&1; tail -n 35 /tmp/t.log` (`run_command` keeps only the first 5,000 chars).\\n- Never inspect outside `/workspace` or install packages (offline container).\\n\\n# CALL PLAN\\n- Calls 1-8: locate and inspect exact target source + caller/test lines.\\n- Calls 9-12: edit SOURCE code. You MUST make at least one source edit by call 12 even if uncertain; an imperfect patch can be refined, pure reading scores 0.\\n- Calls 13-20: run `/tmp` repro + one targeted test file; fix edge cases across all parallel code paths.\\n- Call 21+: stop exploring, verify `git status --short && git diff`, and call `submit_patch`.\\n\\n# WORKFLOW\\n1. **Spec (no tool call)**: List exact files/symbols, error types/messages, return values, and edge cases (`None`, empty, unicode, bounds, defaults, async).\\n2. **Locate**: Start with paths mentioned in the issue. Batch known fixed-string searches in one `run_command` using `git grep -n -F` and `git ls-files` (`head -n 30`). Read 40-120 line slices via `read_file` or `sed -n \'S,Ep\'`, including one caller or sibling test. Keep brief plain-text notes between turns so key facts survive context compaction.\\n3. **Edit**: Use `edit_file` with a short unique `old_string` verbatim from the file. Never repeat an identical failed edit; after 2 match failures, reread exact line numbers and apply one atomic Python replacement via `run_command` quoted heredoc asserting `content.count(old) == 1`. Propagate fixes across ALL parallel implementations (sync/async, `Client`/`AsyncClient`, alternate backends, typing overloads, `__init__.py`/`__all__`, `docs_src/`). Keep unrelated code and signatures intact.\\n4. **Verify**: Run `python3 -m py_compile <edited_file>`, a `/tmp/repro.py` script, and the single closest test file via `/tmp/t.log`. Ignore pre-existing failures from missing optional fixtures or unrelated modules.\\n5. **Submit**: Run `git status --short && git diff`, delete any stray `/workspace` files, revert accidental non-source edits with `git checkout --`, then call `submit_patch` and confirm `patch_size > 0` and `files_changed > 0`. Finish with a one-sentence summary.\\n\\n# DECISION RULES\\n- Ambiguous spec: follow the most literal interpretation and act; never stall.\\n- Stuck on empty searches: try one alternate symbol, then apply your best-reasoned small fix. Never conclude no edit is needed.\\n\\n# Issue\\n{problem_description}\\n"\n}')

for rel_path, content in sorted(SUBMISSION_FILES.items()):
    dest = BUNDLE / rel_path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(content.replace("\r\n", "\n").encode("utf-8"))
    print(f"Wrote {rel_path:24s} ({dest.stat().st_size:,} bytes)")


In [ ]:
ROOT_CONFIGS = ("agent.yaml", "agent.yml", "root_agent.yaml", "root_agent.yml")
ALLOWED_SUFFIXES = {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
ALLOWED_MODELS = (
    "gemma-4-31b-it-qat-w4a16-ct",
    "gemma-4-12b-it-qat-w4a16-ct",
    "gemma-4-e4b-it-qat-w4a16-ct",
    "gemma-4-e2b-it-qat-w4a16-ct",
)
ALLOWED_STATE_VARS = {"problem_description", "hints"}

yaml.SafeLoader.add_constructor("!include", lambda loader, node: loader.construct_scalar(node))

files = sorted(p for p in BUNDLE.rglob("*") if p.is_file())
roots = [name for name in ROOT_CONFIGS if (BUNDLE / name).is_file()]
assert len(roots) == 1, f"Expected 1 root config, found {roots}"
assert not any(p.is_symlink() for p in BUNDLE.rglob("*")), "Symlinks rejected"
assert all(p.suffix in ALLOWED_SUFFIXES for p in files), f"Bad suffix: {[p.name for p in files if p.suffix not in ALLOWED_SUFFIXES]}"
assert sum(p.stat().st_size for p in files) < 3 * 2**30, "Bundle exceeds 3 GiB"

declared_models = set()
for cfg in (p for p in files if p.suffix in (".yaml", ".yml")):
    text = cfg.read_text(encoding="utf-8")
    data = yaml.safe_load(text) or {}
    for inc in re.findall(r"!include\s+(\S+)", text):
        inc_path = (cfg.parent / inc).resolve()
        assert BUNDLE.resolve() in inc_path.parents and inc_path.is_file(), f"{cfg.name}: bad include {inc}"
    assert "adapter" not in data, "Unexpected adapter key"
    if "model" in data:
        declared_models.add(str(data["model"]).strip())

assert len(declared_models) == 1, f"Single model rule violated: {declared_models}"
model = next(iter(declared_models))
assert model in ALLOWED_MODELS, f"Unsupported model: {model}"

for md_file in (p for p in files if p.suffix in (".md", ".txt")):
    placeholders = set(re.findall(r"\{([^}]*)\}", md_file.read_text(encoding="utf-8")))
    unknown = placeholders - ALLOWED_STATE_VARS
    assert not unknown, f"{md_file.name}: unknown ADK state vars {unknown}"

print(f"Schema OK: root={roots[0]} | files={len(files)} | model={model}")


In [ ]:
HARNESS_WHEELS = ("google_adk-", "google_genai-", "adk_submission-", "adk_eval_core-", "swegemma-")
PROVIDER_PREFIXES = ("openai/", "google/", "hosted_vllm/", "custom/")

def install_harness() -> None:
    found = sorted(Path("/kaggle/input").glob("**/adk_submission-*.whl")) if Path("/kaggle/input").exists() else []
    if not found:
        raise FileNotFoundError("metric/gemma-4-developer-agent-wheelhouse not attached")
    wh_dir = found[0].parent
    wheels = [str(w) for w in sorted(wh_dir.glob("*.whl")) if w.name.startswith(HARNESS_WHEELS)]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *wheels], check=True)

def run_command(command: str) -> str: ...
def read_file(filepath: str, start_line: int | None = None, end_line: int | None = None) -> str: ...
def write_file(filepath: str, content: str) -> str: ...
def edit_file(filepath: str, old_string: str, new_string: str, allow_multiple: bool = False) -> str: ...
def submit_patch() -> str: ...
def get_status() -> str: ...

def _norm_model(name: str) -> str:
    name = name.strip().lower()
    for p in PROVIDER_PREFIXES:
        name = name.removeprefix(p)
    return name

ON_KAGGLE = Path("/kaggle/input").exists()
if not ON_KAGGLE:
    try:
        import adk_submission  # noqa: F401
        HAS_HARNESS = True
    except ImportError as exc:
        HAS_HARNESS = False
        print(f"Harness wheel check skipped offline ({exc!r}); structural validation passed.")
else:
    try:
        import adk_submission  # noqa: F401
    except ImportError:
        install_harness()
    HAS_HARNESS = True

if HAS_HARNESS:
    from adk_submission import ModelRegistry, compile_submission, validate_directory
    from adk_submission.discovery import discover_declared_models
    from google.adk.models.lite_llm import LiteLlm
    from swegemma.config import build_submission_limits

    limits, constraints = build_submission_limits()
    validate_directory(BUNDLE, limits)
    print("validate_directory: PASSED")
    declared = discover_declared_models(BUNDLE, normalize_fn=_norm_model)
    assert declared == {model}, f"Mismatch in declared models: {declared}"
    tools = {
        fn.__name__: fn
        for fn in (run_command, read_file, write_file, edit_file, submit_patch, get_status)
    }
    models = ModelRegistry()
    models.register(model, LiteLlm(model=f"openai/{model}", api_base="http://127.0.0.1:9/v1", api_key="EMPTY"))
    compiled_agent = compile_submission(BUNDLE, tools, models, limits=limits, generation_constraints=constraints)
    print(f"compile_submission: PASSED (root_agent={compiled_agent.name!r}, tools={len(tools)})")


In [ ]:
ZIP_PATH = WORKING / "submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w") as zf:
    for p in sorted(f for f in BUNDLE.rglob("*") if f.is_file()):
        info = zipfile.ZipInfo(p.relative_to(BUNDLE).as_posix(), date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        zf.writestr(info, p.read_bytes())

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    assert zf.testzip() is None
    names = zf.namelist()
    assert "agent.yaml" in names and not any(n.endswith("/") for n in names)

shutil.rmtree(BUNDLE)
digest = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print(f"Submission ready: {ZIP_PATH} ({ZIP_PATH.stat().st_size:,} bytes) | SHA-256: {digest}")
for n in names:
    print(" ", n)


## Interpretation
The smaller v3 prompt accompanied a lower observed public score (0.15 versus 0.18). Prompt size, tool selection, temperature and time budget changed together, so these runs cannot identify the effect of prompt compression alone. The compression removed rationale units far more often than directives; whether that matters is a hypothesis for one-factor experiments, not a finding of this notebook. Keep v2 as the best observed candidate.
